# K-Means 聚类可视化

## 目标

理解 K-Means 聚类算法的工作原理。

- 理解 K-Means 的迭代过程
- 从零实现 K-Means 算法
- 选择最佳 K 值的方法
- 可视化聚类过程和结果

## 1. 数据准备

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs, make_circles, make_moons, load_iris
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler
import pandas as pd

np.random.seed(42)

# 生成合成数据
def generate_datasets():
    datasets = {}
    
    # 球形簇数据
    X_blobs, y_blobs = make_blobs(
        n_samples=300, n_features=2, centers=4,
        cluster_std=1.5, random_state=42
    )
    datasets['blobs'] = (X_blobs, y_blobs, '球形簇')
    
    # 不同方差
    X_varied, _ = make_blobs(
        n_samples=300, n_features=2, centers=3,
        cluster_std=[1.0, 2.5, 0.5], random_state=42
    )
    datasets['varied'] = (X_varied, None, '不同方差')
    
    # 各向异性分布
    X_aniso, y_aniso = make_blobs(
        n_samples=300, n_features=2, centers=3,
        cluster_std=1, random_state=42
    )
    transformation = [[0.6, -0.6], [-0.4, 0.8]]
    X_aniso = np.dot(X_aniso, transformation)
    datasets['aniso'] = (X_aniso, y_aniso, '各向异性')
    
    # 环形数据
    X_circles, y_circles = make_circles(
        n_samples=300, factor=0.5, noise=0.05, random_state=42
    )
    datasets['circles'] = (X_circles, y_circles, '环形数据')
    
    return datasets

datasets = generate_datasets()

print("生成的数据集:")
for name, (X, y, desc) in datasets.items():
    print(f"  {name}: {X.shape} - {desc}")

In [ ]:
# 可视化数据集
plt.figure(figsize=(12, 4))

for i, (name, (X, y, desc)) in enumerate(datasets.items()):
    plt.subplot(1, 4, i+1)
    plt.scatter(X[:, 0], X[:, 1], alpha=0.6, s=30)
    plt.xlabel('特征 1')
    plt.ylabel('特征 2')
    plt.title(f'{desc}\n({name})')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. K-Means 算法原理

In [ ]:
class SimpleKMeans:
    """
    从零实现的 K-Means 算法
    """
    
    def __init__(self, n_clusters=3, max_iter=300, tol=1e-4, random_state=None):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.centroids = None
        self.labels_ = None
        self.inertia_ = None
        self.history = []  # 存储迭代历史
    
    def _initialize_centroids(self, X):
        """
        初始化质心（随机选择 K 个样本）
        """
        if self.random_state:
            np.random.seed(self.random_state)
        
        indices = np.random.choice(X.shape[0], self.n_clusters, replace=False)
        return X[indices].copy()
    
    def _assign_clusters(self, X, centroids):
        """
        分配每个样本到最近的质心
        """
        distances = np.zeros((X.shape[0], self.n_clusters))
        
        for i, centroid in enumerate(centroids):
            distances[:, i] = np.linalg.norm(X - centroid, axis=1)
        
        return np.argmin(distances, axis=1)
    
    def _update_centroids(self, X, labels):
        """
        更新质心为簇内样本的均值
        """
        new_centroids = np.zeros((self.n_clusters, X.shape[1]))
        
        for i in range(self.n_clusters):
            cluster_mask = (labels == i)
            if np.any(cluster_mask):
                new_centroids[i] = X[cluster_mask].mean(axis=0)
            else:
                # 如果簇为空，随机重新初始化
                if self.random_state:
                    np.random.seed(self.random_state)
                new_centroids[i] = X[np.random.choice(X.shape[0])]
        
        return new_centroids
    
    def _compute_inertia(self, X, labels, centroids):
        """
        计算簇内平方和（Inertia）
        """
        inertia = 0
        for i in range(self.n_clusters):
            cluster_mask = (labels == i)
            if np.any(cluster_mask):
                inertia += np.sum((X[cluster_mask] - centroids[i]) ** 2)
        return inertia
    
    def fit(self, X):
        """
        训练 K-Means 模型
        """
        # 初始化质心
        self.centroids = self._initialize_centroids(X)
        
        for iteration in range(self.max_iter):
            # 保存旧质心
            old_centroids = self.centroids.copy()
            
            # 分配簇
            self.labels_ = self._assign_clusters(X, self.centroids)
            
            # 更新质心
            self.centroids = self._update_centroids(X, self.labels_)
            
            # 计算 Inertia
            self.inertia_ = self._compute_inertia(X, self.labels_, self.centroids)
            
            # 保存历史
            self.history.append({
                'iteration': iteration,
                'centroids': self.centroids.copy(),
                'labels': self.labels_.copy(),
                'inertia': self.inertia_
            })
            
            # 检查收敛
            centroid_shift = np.linalg.norm(self.centroids - old_centroids)
            if centroid_shift < self.tol:
                break
        
        return self
    
    def predict(self, X):
        """
        预测新样本的簇标签
        """
        return self._assign_clusters(X, self.centroids)
    
    def fit_predict(self, X):
        """
        训练并返回标签
        """
        self.fit(X)
        return self.labels_

print("K-Means 类定义完成")

## 3. 训练自定义 K-Means

In [ ]:
# 使用球形簇数据
X_blobs, y_blobs, _ = datasets['blobs']

# 训练自定义 K-Means
custom_kmeans = SimpleKMeans(n_clusters=4, random_state=42)
custom_kmeans.fit(X_blobs)

print("自定义 K-Means 结果:")
print(f"  迭代次数: {len(custom_kmeans.history)}")
print(f"  最终 Inertia: {custom_kmeans.inertia_:.2f}")
print(f"  簇大小分布: {np.bincount(custom_kmeans.labels_)}")
print(f"  最终质心位置:")
for i, centroid in enumerate(custom_kmeans.centroids):
    print(f"    簇 {i}: ({centroid[0]:.2f}, {centroid[1]:.2f})")

## 4. 可视化 K-Means 迭代过程

In [ ]:
# 绘制迭代过程
n_iterations_to_show = min(6, len(custom_kmeans.history))
selected_iters = [0, 1, 2, 3, len(custom_kmeans.history)-1][:n_iterations_to_show]

plt.figure(figsize=(14, 3))

for i, iter_idx in enumerate(selected_iters):
    plt.subplot(1, n_iterations_to_show, i+1)
    
    centroids = custom_kmeans.history[iter_idx]['centroids']
    labels = custom_kmeans.history[iter_idx]['labels']
    inertia = custom_kmeans.history[iter_idx]['inertia']
    
    # 绘制簇
    colors = ['red', 'blue', 'green', 'orange']
    for j in range(len(centroids)):
        cluster_mask = (labels == j)
        plt.scatter(X_blobs[cluster_mask, 0], X_blobs[cluster_mask, 1], 
                   c=colors[j % len(colors)], alpha=0.6, s=30, label=f'簇 {j}')
    
    # 绘制质心
    plt.scatter(centroids[:, 0], centroids[:, 1], 
               c='black', marker='x', s=100, linewidths=2, label='质心')
    
    plt.xlabel('特征 1')
    plt.ylabel('特征 2')
    plt.title(f'迭代 {iter_idx}\nInertia: {inertia:.0f}')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 绘制 Inertia 收敛曲线
inertias = [h['inertia'] for h in custom_kmeans.history]

plt.figure(figsize=(10, 5))
plt.plot(inertias, 'b-o', linewidth=2, markersize=4)
plt.xlabel('迭代次数')
plt.ylabel('Inertia（簇内平方和）')
plt.title('K-Means 收敛过程')
plt.grid(True, alpha=0.3)
plt.show()

## 5. 与 Scikit-learn 的对比

In [ ]:
# 使用 Scikit-learn 的 K-Means
sklearn_kmeans = KMeans(n_clusters=4, random_state=42)
sklearn_labels = sklearn_kmeans.fit_predict(X_blobs)

print("Scikit-learn K-Means 结果:")
print(f"  迭代次数: {sklearn_kmeans.n_iter_}")
print(f"  Inertia: {sklearn_kmeans.inertia_:.2f}")
print(f"  簇大小分布: {np.bincount(sklearn_labels)}")
print(f"\n与自定义实现对比:")
print(f"  Inertia 差异: {abs(custom_kmeans.inertia_ - sklearn_kmeans.inertia_):.2f}")

In [ ]:
# 可视化对比
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 自定义实现
colors = ['red', 'blue', 'green', 'orange']
for i in range(4):
    cluster_mask = (custom_kmeans.labels_ == i)
    axes[0].scatter(X_blobs[cluster_mask, 0], X_blobs[cluster_mask, 1],
                  c=colors[i], alpha=0.6, s=30)
axes[0].scatter(custom_kmeans.centroids[:, 0], custom_kmeans.centroids[:, 1],
              c='black', marker='x', s=100, linewidths=2)
axes[0].set_xlabel('特征 1')
axes[0].set_ylabel('特征 2')
axes[0].set_title(f'自定义 K-Means\nInertia: {custom_kmeans.inertia_:.0f}')
axes[0].grid(True, alpha=0.3)

# Scikit-learn
for i in range(4):
    cluster_mask = (sklearn_labels == i)
    axes[1].scatter(X_blobs[cluster_mask, 0], X_blobs[cluster_mask, 1],
                  c=colors[i], alpha=0.6, s=30)
axes[1].scatter(sklearn_kmeans.cluster_centers_[:, 0], sklearn_kmeans.cluster_centers_[:, 1],
              c='black', marker='x', s=100, linewidths=2)
axes[1].set_xlabel('特征 1')
axes[1].set_ylabel('特征 2')
axes[1].set_title(f'Scikit-learn K-Means\nInertia: {sklearn_kmeans.inertia_:.0f}')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 在不同数据集上的表现

In [ ]:
# 在所有数据集上测试 K-Means
plt.figure(figsize=(12, 4))

for i, (name, (X, y, desc)) in enumerate(datasets.items()):
    plt.subplot(1, 4, i+1)
    
    # 聚类（假设每个数据集有 2 或 3 个簇）
    n_clusters = 3 if name in ['circles'] else 3
    if name == 'blobs':
        n_clusters = 4
    
    kmeans_temp = KMeans(n_clusters=n_clusters, random_state=42)
    labels_temp = kmeans_temp.fit_predict(X)
    
    # 绘制结果
    colors = ['red', 'blue', 'green', 'orange']
    for j in range(n_clusters):
        cluster_mask = (labels_temp == j)
        if np.any(cluster_mask):
            plt.scatter(X[cluster_mask, 0], X[cluster_mask, 1],
                      c=colors[j % len(colors)], alpha=0.6, s=30)
    
    # 绘制质心
    plt.scatter(kmeans_temp.cluster_centers_[:, 0], kmeans_temp.cluster_centers_[:, 1],
               c='black', marker='x', s=100, linewidths=2)
    
    plt.xlabel('特征 1')
    plt.ylabel('特征 2')
    plt.title(f'{desc}\nInertia: {kmeans_temp.inertia_:.0f}')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n观察:")
print("- 球形簇：K-Means 表现很好")
print("- 不同方差：K-Means 对不同方差的簇不太公平")
print("- 各向异性：K-Means 可能无法正确识别")
print("- 环形数据：K-Means 完全无法识别正确的结构")

## 7. 肘部法则选择最佳 K 值

In [ ]:
# 使用肘部法则寻找最佳 K
K_range = range(1, 11)
inertias = []

for K in K_range:
    kmeans_temp = KMeans(n_clusters=K, random_state=42)
    kmeans_temp.fit(X_blobs)
    inertias.append(kmeans_temp.inertia_)

# 绘制肘部曲线
plt.figure(figsize=(10, 5))
plt.plot(K_range, inertias, 'b-o', linewidth=2, markersize=6)
plt.xlabel('K 值（簇的数量）')
plt.ylabel('Inertia（簇内平方和）')
plt.title('肘部法则：寻找最佳 K 值')
plt.grid(True, alpha=0.3)
plt.xticks(K_range)
plt.show()

# 计算拐点（简化方法：最大曲率）
diff1 = np.diff(inertias)
diff2 = np.diff(diff1)
elbow_point = np.argmax(diff2) + 2  # +2 因为 diff 偏移

print(f"肘部法则建议的 K 值: {elbow_point}")
print(f"\nInertia 值:")
for K, inertia in zip(K_range, inertias):
    marker = " <- 肘部" if K == elbow_point else ""
    print(f"  K={K}: {inertia:.0f}{marker}")

## 8. 轮廓系数选择最佳 K 值

In [ ]:
# 使用轮廓系数评估
K_range = range(2, 11)
silhouette_scores = []

for K in K_range:
    kmeans_temp = KMeans(n_clusters=K, random_state=42)
    labels_temp = kmeans_temp.fit_predict(X_blobs)
    
    # 至少需要 2 个簇才能计算轮廓系数
    if len(np.unique(labels_temp)) > 1:
        score = silhouette_score(X_blobs, labels_temp)
        silhouette_scores.append(score)
    else:
        silhouette_scores.append(-1)

# 绘制轮廓系数曲线
plt.figure(figsize=(10, 5))
plt.plot(K_range, silhouette_scores, 'r-o', linewidth=2, markersize=6)
plt.xlabel('K 值（簇的数量）')
plt.ylabel('平均轮廓系数')
plt.title('轮廓系数：寻找最佳 K 值')
plt.grid(True, alpha=0.3)
plt.xticks(K_range)
plt.axhline(y=0, color='gray', linestyle='--')
plt.show()

# 找到最佳 K
best_K = K_range[np.argmax(silhouette_scores)]
best_score = max(silhouette_scores)

print(f"轮廓系数建议的最佳 K 值: {best_K}")
print(f"最佳轮廓系数: {best_score:.3f}")
print(f"\n轮廓系数解释:")
print("- 接近 1：样本分配正确")
print("- 接近 0：样本在簇边界")
print("- 接近 -1：样本被错误分配")

## 9. 轮廓图可视化

In [ ]:
from sklearn.metrics import silhouette_samples
import matplotlib.cm as cm

# 绘制轮廓图
def plot_silhouette(X, n_clusters):
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(X)
    
    silhouette_vals = silhouette_samples(X, labels)
    y_lower = 10
    
    for i in range(n_clusters):
        ith_silhouette_vals = silhouette_vals[labels == i]
        ith_silhouette_vals.sort()
        size_cluster_i = ith_silhouette_vals.shape[0]
        y_upper = y_lower + size_cluster_i
        
        color = cm.nipy_spectral(float(i) / n_clusters)
        ax.fill_betweenx(np.arange(y_lower, y_upper),
                        0, ith_silhouette_vals,
                        facecolor=color, edgecolor=color, alpha=0.7)
        
        ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10
    
    ax.set_title(f'K={n_clusters} 的轮廓图\n平均轮廓系数={silhouette_score(X, labels):.3f}')
    ax.set_xlabel('轮廓系数')
    ax.set_ylabel('簇')
    ax.set_yticks([])
    ax.axvline(x=silhouette_score(X, labels), color="red", linestyle="--")
    
    plt.tight_layout()
    plt.show()

# 对比不同 K 值的轮廓图
for K in [3, 4, 5]:
    plot_silhouette(X_blobs, K)

## 10. 其他聚类评估指标

In [ ]:
# 对比不同的评估指标
K_range = range(2, 11)

results = {'K': list(K_range),
          'Inertia': [],
          'Silhouette': [],
          'Calinski-Harabasz': [],
          'Davies-Bouldin': []}

for K in K_range:
    kmeans_temp = KMeans(n_clusters=K, random_state=42)
    labels_temp = kmeans_temp.fit_predict(X_blobs)
    
    results['Inertia'].append(kmeans_temp.inertia_)
    results['Silhouette'].append(silhouette_score(X_blobs, labels_temp))
    results['Calinski-Harabasz'].append(calinski_harabasz_score(X_blobs, labels_temp))
    results['Davies-Bouldin'].append(davies_bouldin_score(X_blobs, labels_temp))

# 显示结果
results_df = pd.DataFrame(results)
print("不同 K 值的聚类评估指标:")
print(results_df.round(3).to_string(index=False))

# 可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Inertia
axes[0, 0].plot(K_range, results['Inertia'], 'b-o', linewidth=2, markersize=6)
axes[0, 0].set_xlabel('K 值')
axes[0, 0].set_ylabel('Inertia')
axes[0, 0].set_title('Inertia（越小越好）')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xticks(K_range)

# Silhouette
axes[0, 1].plot(K_range, results['Silhouette'], 'r-o', linewidth=2, markersize=6)
axes[0, 1].axhline(y=0, color='gray', linestyle='--')
axes[0, 1].set_xlabel('K 值')
axes[0, 1].set_ylabel('Silhouette Score')
axes[0, 1].set_title('Silhouette（越大越好）')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xticks(K_range)

# Calinski-Harabasz
axes[1, 0].plot(K_range, results['Calinski-Harabasz'], 'g-o', linewidth=2, markersize=6)
axes[1, 0].set_xlabel('K 值')
axes[1, 0].set_ylabel('Calinski-Harabasz Score')
axes[1, 0].set_title('Calinski-Harabasz（越大越好）')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xticks(K_range)

# Davies-Bouldin
axes[1, 1].plot(K_range, results['Davies-Bouldin'], 'm-o', linewidth=2, markersize=6)
axes[1, 1].set_xlabel('K 值')
axes[1, 1].set_ylabel('Davies-Bouldin Score')
axes[1, 1].set_title('Davies-Bouldin（越小越好）')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xticks(K_range)

plt.tight_layout()
plt.show()

## 11. K-Means++ 初始化

In [ ]:
# 对比随机初始化和 K-Means++ 初始化
n_init_methods = ['random', 'k-means++']

plt.figure(figsize=(12, 5))

for i, init_method in enumerate(n_init_methods):
    plt.subplot(1, 2, i+1)
    
    kmeans_init = KMeans(n_clusters=4, init=init_method, n_init=1, random_state=42)
    labels_init = kmeans_init.fit_predict(X_blobs)
    
    colors = ['red', 'blue', 'green', 'orange']
    for j in range(4):
        cluster_mask = (labels_init == j)
        plt.scatter(X_blobs[cluster_mask, 0], X_blobs[cluster_mask, 1],
                  c=colors[j], alpha=0.6, s=30)
    
    plt.scatter(kmeans_init.cluster_centers_[:, 0], kmeans_init.cluster_centers_[:, 1],
               c='black', marker='x', s=100, linewidths=2)
    
    plt.xlabel('特征 1')
    plt.ylabel('特征 2')
    plt.title(f'{init_method} 初始化\nInertia: {kmeans_init.inertia_:.0f}')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("初始化方法对比:")
print("- random: 随机选择 K 个样本作为初始质心")
print("- k-means++: 智能选择初始质心（默认）")
print("\nK-Means++ 的优势:")
print("  - 减少迭代次数")
print("  - 避免较差的局部最优")

## 12. K-Means 的局限性

In [ ]:
# 演示 K-Means 的局限性

# 1. 对异常值敏感
X_outliers = np.vstack([X_blobs, np.array([[10, 10], [-10, -10]])])

plt.figure(figsize=(14, 4))

# 正常数据
plt.subplot(1, 3, 1)
kmeans_normal = KMeans(n_clusters=4, random_state=42)
labels_normal = kmeans_normal.fit_predict(X_blobs)
colors = ['red', 'blue', 'green', 'orange']
for j in range(4):
    cluster_mask = (labels_normal == j)
    plt.scatter(X_blobs[cluster_mask, 0], X_blobs[cluster_mask, 1],
                  c=colors[j], alpha=0.6, s=30)
plt.scatter(kmeans_normal.cluster_centers_[:, 0], kmeans_normal.cluster_centers_[:, 1],
           c='black', marker='x', s=100, linewidths=2)
plt.xlabel('特征 1')
plt.ylabel('特征 2')
plt.title('正常数据')
plt.grid(True, alpha=0.3)

# 有异常值
plt.subplot(1, 3, 2)
kmeans_outliers = KMeans(n_clusters=4, random_state=42)
labels_outliers = kmeans_outliers.fit_predict(X_outliers)
for j in range(4):
    cluster_mask = (labels_outliers == j)
    plt.scatter(X_outliers[cluster_mask, 0], X_outliers[cluster_mask, 1],
                  c=colors[j], alpha=0.6, s=30)
# 标记异常值
plt.scatter([10, -10], [10, -10], c='yellow', marker='*', s=200, 
          edgecolors='red', linewidths=2, label='异常值')
plt.scatter(kmeans_outliers.cluster_centers_[:, 0], kmeans_outliers.cluster_centers_[:, 1],
           c='black', marker='x', s=100, linewidths=2)
plt.xlabel('特征 1')
plt.ylabel('特征 2')
plt.title('有异常值')
plt.legend()
plt.grid(True, alpha=0.3)

# 环形数据
plt.subplot(1, 3, 3)
X_circles, _, _ = datasets['circles']
kmeans_circles = KMeans(n_clusters=2, random_state=42)
labels_circles = kmeans_circles.fit_predict(X_circles)
colors_circle = ['red', 'blue']
for j in range(2):
    cluster_mask = (labels_circles == j)
    plt.scatter(X_circles[cluster_mask, 0], X_circles[cluster_mask, 1],
                  c=colors_circle[j], alpha=0.6, s=30)
plt.scatter(kmeans_circles.cluster_centers_[:, 0], kmeans_circles.cluster_centers_[:, 1],
           c='black', marker='x', s=100, linewidths=2)
plt.xlabel('特征 1')
plt.ylabel('特征 2')
plt.title('环形数据（无法正确聚类）')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("K-Means 的局限性:")
print("1. 对异常值敏感：异常值会拉偏质心")
print("2. 假设簇是球形的：无法识别非球形簇")
print("3. 需要预先指定 K 值：不知道最佳簇数")
print("4. 对初始质心敏感：可能陷入局部最优")
print("5. 对尺度敏感：需要标准化数据")

## 13. 实际应用：Iris 数据集聚类

In [ ]:
# 在 Iris 数据集上应用 K-Means
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
feature_names_iris = iris.feature_names
target_names_iris = iris.target_names

# 标准化数据
scaler_iris = StandardScaler()
X_iris_scaled = scaler_iris.fit_transform(X_iris)

# 聚类
kmeans_iris = KMeans(n_clusters=3, random_state=42)
labels_iris = kmeans_iris.fit_predict(X_iris_scaled)

print(f"Iris 数据集聚类结果:")
print(f"  Inertia: {kmeans_iris.inertia_:.2f}")
print(f"  簇大小分布: {np.bincount(labels_iris)}")
print(f"  轮廓系数: {silhouette_score(X_iris_scaled, labels_iris):.3f}")

# 对比聚类结果与真实标签
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(y_iris, labels_iris)
nmi = normalized_mutual_info_score(y_iris, labels_iris)

print(f"\n与真实标签的相似度:")
print(f"  ARI（调整兰德指数）: {ari:.3f}")
print(f"  NMI（标准化互信息）: {nmi:.3f}")

In [ ]:
# 可视化聚类结果（使用前两个特征）
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 真实标签
for i, target_name in enumerate(target_names_iris):
    mask = (y_iris == i)
    axes[0].scatter(X_iris_scaled[mask, 0], X_iris_scaled[mask, 1],
                  label=target_name, alpha=0.7)
axes[0].set_xlabel(feature_names_iris[0])
axes[0].set_ylabel(feature_names_iris[1])
axes[0].set_title('真实标签')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 聚类结果
colors = ['red', 'blue', 'green']
for i in range(3):
    mask = (labels_iris == i)
    axes[1].scatter(X_iris_scaled[mask, 0], X_iris_scaled[mask, 1],
                  c=colors[i], label=f'簇 {i}', alpha=0.7)
axes[1].set_xlabel(feature_names_iris[0])
axes[1].set_ylabel(feature_names_iris[1])
axes[1].set_title(f'K-Means 聚类\nARI={ari:.3f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 总结

### K-Means 算法步骤

1. **初始化**：随机选择 K 个样本作为初始质心
2. **分配**：将每个样本分配到最近的质心
3. **更新**：重新计算每个簇的质心（均值）
4. **重复**：重复步骤 2-3 直到收敛

### K-Means 的优缺点

**优点**:
- 简单易懂，易于实现
- 计算效率高
- 适合球形簇
- 可扩展性良好

**缺点**:
- 需要预先指定 K 值
- 对初始质心敏感
- 对异常值敏感
- 假设簇是球形的
- 无法识别非凸形状的簇

### 选择最佳 K 值的方法

| 方法 | 指标 | 最佳值 | 特点 |
|------|------|--------|------|
| 肘部法则 | Inertia | 拐点 | 直观，但不够精确 |
| 轮廓系数 | 平均轮廓系数 | 最大值 | 考虑簇内紧密度和簇间分离度 |
| Gap Statistic | Gap | 最大值 | 更严格，计算复杂 |

### 实践建议

- 始终先对数据进行标准化
- 使用 K-Means++ 初始化（默认）
- 运行多次（n_init > 1）以避免局部最优
- 结合多种方法选择 K 值
- 使用轮廓图验证聚类质量
- 考虑异常值的影响
- 对于非球形数据，考虑其他聚类算法（如 DBSCAN、层次聚类）